In [23]:
import os
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest


In [24]:
BASE_PATH = "data/raw"
OUTPUT_PATH = "data/processed"

os.makedirs(OUTPUT_PATH, exist_ok=True)


In [25]:
client_train = pd.read_csv(os.path.join(BASE_PATH, "client_train.csv"), low_memory=False)
invoice_train = pd.read_csv(os.path.join(BASE_PATH, "invoice_train.csv"), low_memory=False)


In [26]:
# 2. Clean column names
client_train.columns = (
    client_train.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

invoice_train.columns = (
    invoice_train.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

In [28]:
# 3. Merge invoice + client
df = invoice_train.merge(client_train, on="client_id", how="left")


In [29]:
#Convert date
df["invoice_date"] = pd.to_datetime(df["invoice_date"], errors="coerce")

In [31]:
# 5. Create total consumption column
consumption_cols = [
    "consommation_level_1",
    "consommation_level_2",
    "consommation_level_3",
    "consommation_level_4"
]

for col in consumption_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

df["consumption_units"] = df[consumption_cols].sum(axis=1)

In [32]:
# 7. Feature engineering
df = df.sort_values(["client_id", "invoice_date"])

df["previous_consumption"] = (
    df.groupby("client_id")["consumption_units"]
    .shift(1)
    .fillna(0)
)

df["average_3m_consumption"] = (
    df.groupby("client_id")["consumption_units"]
    .rolling(3, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

df["consumption_change_percent"] = (
    (df["consumption_units"] - df["previous_consumption"]) /
    df["previous_consumption"].replace(0, np.nan)
) * 100

df["consumption_change_percent"] = (
    df["consumption_change_percent"]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

df["is_zero_consumption"] = np.where(df["consumption_units"] == 0, 1, 0)
df["is_negative_consumption"] = np.where(df["consumption_units"] < 0, 1, 0)

In [33]:
# 8.Rule based anamoly Detection
#=============================================================
df["rule_anomaly_flag"] = np.where(
    (df["consumption_units"] < 0) |
    (df["consumption_units"] == 0) |
    (df["consumption_units"] > df["average_3m_consumption"] * 3) |
    (df["consumption_units"] < df["average_3m_consumption"] * 0.2),
    1,
    0
)

In [34]:
# 9. Machine Learning Anomaly Detection - Isolation Forest
# ============================================================

features = [
    "consumption_units",
    "previous_consumption",
    "average_3m_consumption",
    "consumption_change_percent",
    "is_zero_consumption",
    "is_negative_consumption"
]

X = df[features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42
)

df["model_anomaly_prediction"] = model.fit_predict(X_scaled)

df["model_anomaly_flag"] = np.where(
    df["model_anomaly_prediction"] == -1,
    1,
    0
)

df["anomaly_score"] = model.decision_function(X_scaled)


In [35]:
# 10. Final Anomaly Flag
# ============================================================

df["final_anomaly_flag"] = np.where(
    (df["rule_anomaly_flag"] == 1) |
    (df["model_anomaly_flag"] == 1),
    1,
    0
)

In [36]:
# 11. Create Anomaly Type
# ============================================================

df["anomaly_type"] = np.select(
    [
        df["consumption_units"] < 0,
        df["consumption_units"] == 0,
        df["consumption_units"] > df["average_3m_consumption"] * 3,
        df["consumption_units"] < df["average_3m_consumption"] * 0.2,
        df["model_anomaly_flag"] == 1
    ],
    [
        "Negative Consumption",
        "Zero Consumption",
        "High Consumption Spike",
        "Sudden Consumption Drop",
        "Model Detected Anomaly"
    ],
    default="Normal"
)

In [ ]:
# 12. Save Final Processed Dataset
# ============================================================

output_file = os.path.join(
    OUTPUT_PATH,
    "utility_anomaly_detection_output.csv"
)

df.to_csv(output_file, index=False)

In [ ]:
# 13. Summary Output
# ============================================================

print("Anomaly detection completed successfully.")
print("Processed file saved at:", output_file)
print()
print("Final anomaly flag count:")
print(df["final_anomaly_flag"].value_counts())
print()
print("Anomaly type count:")
print(df["anomaly_type"].value_counts())
print()
print("Final dataset shape:", df.shape)